# Indian E-commerce Returns & Fulfilment Analysis
## Data Cleaning and Preparation

**Purpose:** Create one clean, compact order-item dataset for EDA, SQL Server, and Power BI. Each row represents one order item.

**Source:** Kaggle — `abirambharathisa/india-e-commerce-orders-and-returns-2024-2026`.


## 1. Import Libraries

In [ ]:
import pandas as pd
from pathlib import Path

OUTPUT_PATH = Path("cleaned_ecommerce_order_items.csv")


Choose one method. For a local download, keep the CSV beside this notebook.

For Kaggle: first run `%pip install -q kagglehub` in a separate notebook cell, then set `LOAD_METHOD = "kaggle"`. Kaggle access may require account setup.


In [ ]:
LOAD_METHOD = "local"  # Change to "kaggle" to download from Kaggle

if LOAD_METHOD == "local":
    DATA_PATH = Path("india_ecommerce_orders_returns.csv")
    if not DATA_PATH.exists():
        raise FileNotFoundError("Place india_ecommerce_orders_returns.csv beside this notebook.")
else:
    import kagglehub
    dataset_path = Path(kagglehub.dataset_download(
        "abirambharathisa/india-e-commerce-orders-and-returns-2024-2026"
    ))
    csv_files = list(dataset_path.rglob("india_ecommerce_orders_returns.csv"))
    if not csv_files:
        csv_files = list(dataset_path.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError("No CSV file found in the Kaggle dataset.")
    DATA_PATH = csv_files[0]

# Some source records are malformed; the parser skips those records.
df = pd.read_csv(DATA_PATH, engine="python", on_bad_lines="skip")
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")


## 3. Inspect the Raw Data

In [ ]:
display(df.head())
print("Raw shape:", df.shape)


## 4. Standardize Columns and Text

Standardize column names and remove extra spaces from text fields.

In [ ]:
df.columns = (
    df.columns.str.strip().str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()


## 5. Keep Relevant Columns

Keep only fields used for the project's sales, customer, product, returns, refunds, and delivery analysis. Order-level totals (`order_value` and `amount_paid`) are excluded because they repeat across item rows. Highly sparse or non-essential fields are also excluded.

In [ ]:
keep_cols = [
    "order_id", "order_item_id", "order_datetime", "customer_id",
    "state", "region", "is_plus_member", "payment_method",
    "category", "sub_category", "brand", "product_id", "seller_id",
    "seller_rating", "quantity", "mrp", "discount_pct", "selling_price",
    "item_total", "courier_partner", "promised_delivery_days",
    "shipped_datetime", "delivered_datetime", "actual_delivery_days",
    "is_late_delivery", "order_status", "cancel_reason", "item_status",
    "is_returned", "return_reason", "return_status", "refund_mode",
    "refund_amount", "refund_days"
]

missing_columns = [col for col in keep_cols if col not in df.columns]
if missing_columns:
    raise KeyError(f"Expected columns missing from source: {missing_columns}")

df = df[keep_cols].copy()
print(f"Retained {len(df.columns)} relevant columns")


## 6. Fix Data Types and Remove Exact Duplicates

In [ ]:
date_cols = ["order_datetime", "shipped_datetime", "delivered_datetime"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

numeric_cols = [
    "seller_rating", "quantity", "mrp", "discount_pct", "selling_price",
    "item_total", "promised_delivery_days", "actual_delivery_days",
    "is_late_delivery", "refund_amount", "refund_days"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Store binary flags as 0/1 for consistent pandas and SQL Server use.
for col in ["is_plus_member", "is_returned"]:
    values = df[col].astype("string").str.strip().str.lower()
    df[col] = values.map({"true": 1, "false": 0, "1": 1, "0": 0})

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("Exact duplicate rows removed:", before - len(df))
print("Final working shape:", df.shape)


## 7. Check Missing Values and Key Data Quality

Missing delivery and return fields may be expected when an item has not been delivered or returned. Do not fill these values blindly.

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
})
display(missing[missing["missing_count"] > 0].sort_values("missing_percent", ascending=False))

checks = pd.Series({
    "Rows": len(df),
    "Columns": len(df.columns),
    "Missing order IDs": df["order_id"].isna().sum(),
    "Missing order-item IDs": df["order_item_id"].isna().sum(),
    "Duplicate order-item keys": df.duplicated(["order_id", "order_item_id"]).sum(),
    "Non-positive quantities": (df["quantity"] <= 0).sum(),
    "Discount outside 0-100%": ((df["discount_pct"] < 0) | (df["discount_pct"] > 100)).sum()
})
display(checks.to_frame("result"))


## 8. Export the Cleaned Dataset

In [ ]:
df.to_csv(OUTPUT_PATH, index=False, date_format="%Y-%m-%d %H:%M:%S")
print(f"Saved: {OUTPUT_PATH.resolve()}")
print(f"Final shape: {df.shape}")
print("Use this same CSV for the EDA notebook, SQL Server, and Power BI.")
